In [ ]:
# rag_pipeline.py
from sentence_transformers import CrossEncoder


In [22]:
reranker = CrossEncoder("BAAI/bge-reranker-base")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

c:\Users\조영석\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\조영석\.cache\huggingface\hub\models--BAAI--bge-reranker-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

In [ ]:
def rerank(question, docs, top_k=3):
    # 1. 질문이랑 각 문서를 '쌍'으로 묶기
    pairs = [(question,doc) for doc in docs]                          # [(질문, 문서1), (질문, 문서2), ...]
    # 2. 각 쌍의 관련도 점수 매기기
    scores = reranker.predict(pairs)     # 점수 배열
    # 3. 점수 높은 순으로 문서 정렬 → 상위 top_k개 반환
    scored = list(zip(scores, docs))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for score, doc in scored[:top_k]]




6


In [25]:
from rag_app import build_vectorstore, TEXTS

vectorstore = build_vectorstore(TEXTS)

def retrieve_and_rerank(question, first_k=20, top_k=5):
    # 1. 검색: 후보를 넉넉히 (first_k개)
    candidates = vectorstore.similarity_search(question, k=first_k)   # Document 리스트
    # 2. 재정렬: candidates에서 상위 top_k로 좁히기
    texts = [d.page_content for d in candidates]
    reranked = rerank(question, texts, top_k)
    return reranked

print(vectorstore._collection.count())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

18


In [9]:
q = "RAG는 환각을 어떻게 줄이는가?"

# 재정렬 전 (그냥 검색 순서)
before = [d.page_content for d in vectorstore.similarity_search(q, k=5)]
# 재정렬 후
after = retrieve_and_rerank(q, first_k=5, top_k=3)

print("=== 검색 순서 (before) ===")
for i, t in enumerate(before, 1):
    print(f"{i}. {t}")
print("\n=== 재정렬 후 (after) ===")
for i, t in enumerate(after, 1):
    print(f"{i}. {t}")

=== 검색 순서 (before) ===
1. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
2. 파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.
3. 청크 크기가 너무 작으면 문장이 토막나 검색 품질이 떨어지고, 너무 크면 잡내용이 섞여 흐려진다.
4. 벡터 데이터베이스는 임베딩을 저장하고 코사인 유사도로 가까운 것을 빠르게 검색한다.
5. 임베딩은 텍스트를 의미를 담은 고차원 벡터로 바꾼 것이다.

=== 재정렬 후 (after) ===
1. RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.
2. 벡터 데이터베이스는 임베딩을 저장하고 코사인 유사도로 가까운 것을 빠르게 검색한다.
3. 파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

llm = ChatGroq(model="openai/gpt-oss-120b")   # 생성용 (또는 gpt-oss-120b)
prompt = ChatPromptTemplate.from_template(
    """아래 컨텍스트를 근거로만 질문에 답하라. 없으면 없다고 답하라. 한국어로 답하라.

컨텍스트:
{context}

질문: {question}

답변:"""
)

def generate_answer(question, reranked_docs):
    context = "\n\n".join(reranked_docs)   # 재정렬된 문서들을 하나의 문자열로
    messages = prompt.format_messages(context=context, question=question)
    response = llm.invoke(messages)
    return response.content

In [17]:
def rag_pipeline(question):
    reranked = retrieve_and_rerank(question, first_k=5, top_k=3)   # 검색+재정렬
    answer = generate_answer(question, reranked)                    # 생성
    return answer, reranked   # 답 + 근거(평가에 쓸 거)

In [18]:
q = "RAG는 환각을 어떻게 줄이는가?"
answer, docs = rag_pipeline(q)
print("답:", answer)
print("근거:", docs)

답: RAG는 질문에 대한 답변을 만들 때, 사전에 검색한 관련 문서를 프롬프트에 포함시킵니다. 이렇게 외부의 실제 문서 내용을 직접 참조하도록 함으로써 모델이 스스로 추론하거나 추측해 내는(환각) 가능성을 낮추고, 최신·정확한 정보를 답변에 반영합니다.
근거: ['RAG는 관련 문서를 검색해 프롬프트에 넣어줌으로써 환각을 줄이고 최신 정보를 답에 반영한다.', '벡터 데이터베이스는 임베딩을 저장하고 코사인 유사도로 가까운 것을 빠르게 검색한다.', '파인튜닝은 사전학습된 모델의 가중치를 특정 데이터로 추가 학습해 행동을 바꾸는 것이다.']


In [ ]:
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import faithfulness
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# judge (gpt-oss-120b로!)
judge_llm = LangchainLLMWrapper(ChatGroq(model="openai/gpt-oss-120b"))
judge_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")
)

# 파이프라인으로 평가 샘플 모으기  ← 네가 채울 부분
questions = ["RAG는 환각을 어떻게 줄이는가?",
             "파인튜닝을 하는 이유는 무엇인가?",
             "LLM이 정보를 검색할 때의 과정은 어떻게 되는가?"]   # 아까 그 질문 3개 재활용
samples = []
for q in questions:
    answer, docs = rag_pipeline(q)          # ← 재정렬 파이프라인 사용!
    samples.append(SingleTurnSample(
        user_input=q,
        retrieved_contexts=docs,             # 이미 문자열 리스트 (rerank 결과)
        response=answer,
    ))

dataset = EvaluationDataset(samples=samples)
result = evaluate(dataset=dataset, metrics=[faithfulness], llm=judge_llm, embeddings=judge_emb)
print(result)

C:\Users\조영석\AppData\Local\Temp\ipykernel_22284\2677985142.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
C:\Users\조영석\AppData\Local\Temp\ipykernel_22284\2677985142.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  judge_llm = LangchainLLMWrapper(ChatGroq(model="openai/gpt-oss-120b"))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\조영석\AppData\Local\Temp\ipykernel_22284\2677985142.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  judge_emb = LangchainEmbeddingsWrapper(


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

{'faithfulness': 1.0000}
